# Diabetes Risk Prediction Project

This notebook builds a diabetes risk screening model using a public Pima dataset.

**Important:** This is decision-support only, not a medical diagnosis.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    classification_report, confusion_matrix, f1_score, precision_score,
    recall_score, brier_score_loss
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')

In [ ]:
# Public Pima dataset mirror (CSV)
DATA_URL = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
df = pd.read_csv(DATA_URL)

print('Shape:', df.shape)
print('\nColumns:', list(df.columns))
display(df.head())
display(df.describe())

In [ ]:
# Basic quality audit: some physiological measures cannot be 0 in practice.
target_col = 'Outcome'
zero_invalid_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

audit = []
for c in zero_invalid_cols:
    audit.append({'feature': c, 'zero_count': int((df[c] == 0).sum()), 'zero_pct': float((df[c] == 0).mean())})
audit_df = pd.DataFrame(audit).sort_values('zero_pct', ascending=False)
display(audit_df)

print('Class balance:')
display(df[target_col].value_counts(normalize=True).rename('proportion'))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col)
plt.title('Target Distribution')
plt.show()

In [ ]:
# Convert invalid zeros to missing values before imputation.
df_model = df.copy()
for c in zero_invalid_cols:
    df_model[c] = df_model[c].replace(0, np.nan)

X = df_model.drop(columns=[target_col])
y = df_model[target_col]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train_val
)

print('Train shape:', X_train.shape)
print('Validation shape:', X_val.shape)
print('Test shape:', X_test.shape)

In [ ]:
numeric_features = X.columns.tolist()

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocess = ColumnTransformer([
    ('num', numeric_pipe, numeric_features)
])

models = {
    'logistic_regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(n_estimators=400, min_samples_leaf=2, class_weight='balanced', random_state=RANDOM_STATE),
    'gradient_boosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['roc_auc', 'average_precision', 'f1', 'precision', 'recall']

cv_rows = []
for name, model in models.items():
    pipe = Pipeline([
        ('preprocess', preprocess),
        ('model', model)
    ])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        'model': name,
        'cv_roc_auc': scores['test_roc_auc'].mean(),
        'cv_pr_auc': scores['test_average_precision'].mean(),
        'cv_f1': scores['test_f1'].mean(),
        'cv_precision': scores['test_precision'].mean(),
        'cv_recall': scores['test_recall'].mean()
    })

cv_results = pd.DataFrame(cv_rows).sort_values(['cv_roc_auc', 'cv_recall'], ascending=False)
display(cv_results)

In [ ]:
best_model_name = cv_results.iloc[0]['model']
print('Selected model from CV:', best_model_name)

base_pipeline = Pipeline([
    ('preprocess', preprocess),
    ('model', models[best_model_name])
])
base_pipeline.fit(X_train, y_train)

# Calibrate probabilities on training data only via internal CV.
calibrated_model = CalibratedClassifierCV(estimator=base_pipeline, cv=5, method='sigmoid')
calibrated_model.fit(X_train, y_train)

val_proba = calibrated_model.predict_proba(X_val)[:, 1]
val_roc = roc_auc_score(y_val, val_proba)
val_pr = average_precision_score(y_val, val_proba)
print(f'Validation ROC-AUC: {val_roc:.4f}')
print(f'Validation PR-AUC:  {val_pr:.4f}')

prec, rec, thr = precision_recall_curve(y_val, val_proba)
thr = np.append(thr, 1.0)

candidate = pd.DataFrame({'threshold': thr, 'precision': prec, 'recall': rec})
candidate['f1'] = (2 * candidate['precision'] * candidate['recall']) / (candidate['precision'] + candidate['recall'] + 1e-9)

# Recall-prioritized threshold selection for screening use.
min_recall = 0.80
eligible = candidate[candidate['recall'] >= min_recall]
if len(eligible) == 0:
    selected_threshold = float(candidate.sort_values('f1', ascending=False).iloc[0]['threshold'])
else:
    selected_threshold = float(eligible.sort_values('f1', ascending=False).iloc[0]['threshold'])

print(f'Selected threshold: {selected_threshold:.3f}')

In [ ]:
test_proba = calibrated_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= selected_threshold).astype(int)

test_metrics = {
    'roc_auc': roc_auc_score(y_test, test_proba),
    'pr_auc': average_precision_score(y_test, test_proba),
    'f1': f1_score(y_test, test_pred),
    'precision': precision_score(y_test, test_pred),
    'recall': recall_score(y_test, test_pred),
    'brier_score': brier_score_loss(y_test, test_proba)
}
display(pd.DataFrame([test_metrics]))

print('Classification report (test):')
print(classification_report(y_test, test_pred, digits=4))

cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Test)')
plt.show()

In [ ]:
# Risk bands for practical triage interpretation.
def risk_band(p):
    if p < 0.33:
        return 'Low'
    if p < 0.66:
        return 'Medium'
    return 'High'

risk_df = X_test.copy()
risk_df['actual_outcome'] = y_test.values
risk_df['predicted_probability'] = test_proba
risk_df['predicted_class'] = test_pred
risk_df['risk_band'] = risk_df['predicted_probability'].apply(risk_band)

display(risk_df[['actual_outcome', 'predicted_probability', 'predicted_class', 'risk_band']].head(10))
display(risk_df['risk_band'].value_counts(normalize=True).rename('proportion'))

In [ ]:
# Global explainability with permutation importance.
perm = permutation_importance(
    calibrated_model,
    X_test,
    y_test,
    scoring='roc_auc',
    n_repeats=25,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
display(importance_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df.head(8), x='importance_mean', y='feature')
plt.title('Top Permutation Importances (ROC-AUC impact)')
plt.xlabel('Mean importance')
plt.ylabel('Feature')
plt.show()

In [ ]:
# Local explanation snapshots: low/medium/high risk examples.
sample_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'Age', 'predicted_probability', 'risk_band', 'actual_outcome']
for band in ['Low', 'Medium', 'High']:
    subset = risk_df[risk_df['risk_band'] == band].sort_values('predicted_probability', ascending=False)
    print(f'\n{band} risk example(s):')
    display(subset[sample_cols].head(3))

print('Notebook complete. Use these outputs in diabetes_risk_report.md.')